# Day 1 — Data and Mathematical Foundation

## Goal

Understand how customer data becomes mathematical input for Logistic Regression.

## Core Idea

X = input features

y = target

For one customer:

X = [x₁, x₂, ..., xₙ]

The linear model calculates:

z = wᵀX + b

Where:

- X = feature vector
- w = weight vector
- b = bias
- z = linear combination

In [5]:
import pandas as pd

df = pd.read_csv("../data/dataset.csv")

df.head()

,age,tenure,monthly_charges,support_calls,contract_type,churn
0,22,2,75,5,month-to-month,1
1,45,36,55,1,two-year,0
2,31,8,90,4,month-to-month,1
3,52,48,60,1,two-year,0
4,28,5,85,3,month-to-month,1


In [6]:


X = df.drop("churn", axis=1)
y = df["churn"]

print("Features (X):")
print(X)

print("\nTarget (y):")
print(y)

Features (X):
    age  tenure  monthly_charges  support_calls   contract_type
0    22       2               75              5  month-to-month
1    45      36               55              1        two-year
2    31       8               90              4  month-to-month
3    52      48               60              1        two-year
4    28       5               85              3  month-to-month
5    39      24               65              2        one-year
6    61      60               50              0        two-year
7    25       4               95              6  month-to-month
8    47      30               58              1        one-year
9    34      12               88              4  month-to-month
10   56      42               62              1        two-year
11   29       7               78              5  month-to-month
12   41      18               70              2        one-year
13   63      55               52              0        two-year
14   36      10           

In [7]:
x = [22, 2, 75, 5]
w = [0.1, 0.2, 0.05, 0.3]
b = 1

z = (
    w[0] * x[0]
    + w[1] * x[1]
    + w[2] * x[2]
    + w[3] * x[3]
    + b
)

print("z =", z)

z = 8.85


In [8]:
import numpy as np

x = np.array([22, 2, 75, 5])
w = np.array([0.1, 0.2, 0.05, 0.3])
b = 1

z = np.dot(w, x) + b

print("wᵀx + b =", z)

wᵀx + b = 8.850000000000001


## Linear Model

For one customer, our numerical feature vector is:

x = [x₁, x₂, ..., xₙ]

The model has a weight vector:

w = [w₁, w₂, ..., wₙ]

The linear model calculates:

z = wᵀx + b

The dot product is:

wᵀx = w₁x₁ + w₂x₂ + ... + wₙxₙ

For our example:

x = [22, 2, 75, 5]

w = [0.1, 0.2, 0.05, 0.3]

b = 1

Therefore:

z = (0.1×22) + (0.2×2) + (0.05×75) + (0.3×5) + 1

z = 8.85

### Important

The model has not learned these weights yet.

We selected them manually only to understand the mathematics.

Later, gradient descent will learn the weights from the data.

## Numerical vs Categorical Features

Our dataset contains two types of features.

### Numerical features

These already have numerical values:

- age
- tenure
- monthly_charges
- support_calls

Example:

x = [22, 2, 75, 5]

These can directly participate in mathematical operations.

### Categorical feature

contract_type contains categories:

- month-to-month
- one-year
- two-year

These cannot directly be multiplied by numerical weights.

Therefore, we will transform the categorical feature into a numerical representation.

We will learn and implement this encoding during Day 2.

### Important

We should not simply assign:

month-to-month = 1
one-year = 2
two-year = 3

without understanding the consequences, because those numbers could incorrectly suggest an ordered relationship.


## Dimensions of X and w

Our dataset contains:

- 20 customers
- 4 numerical features used for today's mathematical example

Therefore:

X ∈ R^(20×4)

For one customer:

x ∈ R^4

The weight vector contains one weight for every feature:

w ∈ R^4

Therefore the dot product is valid:

wᵀx

because both x and w contain 4 corresponding values.

The linear model is:

z = wᵀx + b

The model has one weight for each input feature.

In [9]:
print("X shape:", X.shape)

x_numeric = X[["age", "tenure", "monthly_charges", "support_calls"]].iloc[0].to_numpy()

print("One customer x:", x_numeric)
print("Number of features:", len(x_numeric))

X shape: (20, 5)
One customer x: [22  2 75  5]
Number of features: 4


In [10]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import load_data

dataset_path = PROJECT_ROOT / "data" / "dataset.csv"

df_test = load_data(dataset_path)

print("Shape:", df_test.shape)
print(df_test.head())

Shape: (20, 6)
   age  tenure  monthly_charges  support_calls   contract_type  churn
0   22       2               75              5  month-to-month      1
1   45      36               55              1        two-year      0
2   31       8               90              4  month-to-month      1
3   52      48               60              1        two-year      0
4   28       5               85              3  month-to-month      1


# Day 2 — Preprocessing and Logistic Regression Mathematics


## Goal

Transform raw dataset features into standardized numerical inputs, then apply the Sigmoid function to output probabilities.


## Core Idea

Machine learning models only understand numbers, and they learn best when those numbers are on a similar scale.

->Train/Test Split: Isolate a test set to evaluate real-world performance later.

->Encoding: Convert text categories (like "month-to-month") into 0s and 1s.

->Standardization: x−μ/σ

->Sigmoid: σ(z)= 1/1+e^(−z)

In [11]:
import numpy as np
import pandas as pd
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

# Import the functions you wrote!
from src.preprocessing import (
    load_data, train_test_split, encode_categorical, 
    calculate_mean, calculate_std, standardize
)
from src.logistic_regression import (
    sigmoid, predict_probability, probability_to_class
)

dataset_path = PROJECT_ROOT / "data" / "dataset.csv"
df = load_data(dataset_path)

X = df.drop("churn", axis=1)
y = df["churn"]

## 1. Train / Test Split

If a student sees the exam paper before the test, getting a 100% doesn't mean they are smart—it just means they memorized it! The same applies to Machine Learning. We must hide some data (the test set) so we can evaluate the model fairly later.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training features shape:", X_train.shape)
print("Testing features shape:", X_test.shape)

Training features shape: (16, 5)
Testing features shape: (4, 5)


## 2. Categorical Encoding (One-Hot Encoding)
In Day 1, we saw that contract_type contains text like "month-to-month" or "two-year". You cannot multiply a weight w 
i
​	
  by "month-to-month"!

We could just assign 1, 2, and 3 to them. But the model would think that "two-year" (3) is mathematically greater or worth more than "month-to-month" (1). Instead, we use your encode_categorical function to create binary flags.

In [13]:
X_train_encoded, X_test_encoded = encode_categorical(X_train, X_test)

print("Original Data (First 2 rows):")
print(X_train[["age", "contract_type"]].head(2))

print("\nEncoded Data (First 2 rows):")
print(X_train_encoded[["age", "month-to-month", "one-year", "two-year"]].head(2))

Original Data (First 2 rows):
    age contract_type
12   41      one-year
10   56      two-year

Encoded Data (First 2 rows):
    age  month-to-month  one-year  two-year
12   41               0         1         0
10   56               0         0         1


## 3. Feature Standardization
Right now, age might be around 40, but monthly_charges might be 90. If we do not scale these, the gradients for larger features will bounce around wildly during gradient descent.

We standardize them using the training mean μ and standard deviation σ:

x(scaled) = x−μ / σ
​	

Important: We must calculate the mean and standard deviation strictly on the training set, and use those exact same values to scale the test set!

In [14]:
# Convert to NumPy for mathematical operations
X_train_num = X_train_encoded.to_numpy(dtype=float)
X_test_num = X_test_encoded.to_numpy(dtype=float)
y_train_num = y_train.to_numpy(dtype=int)

# Calculate mu and sigma on training data
train_mean = calculate_mean(X_train_num)
train_std = calculate_std(X_train_num, train_mean)

# Scale both sets
X_train_scaled = standardize(X_train_num, train_mean, train_std)
X_test_scaled = standardize(X_test_num, train_mean, train_std)

print("First customer BEFORE scaling:")
print(np.round(X_train_num[0], 2))

print("\nFirst customer AFTER scaling:")
print(np.round(X_train_scaled[0], 2))

First customer BEFORE scaling:
[41. 18. 70.  2.  0.  1.  0.]

First customer AFTER scaling:
[-0.1  -0.47  0.08 -0.17 -0.77  1.73 -0.77]


## 4. The Sigmoid Function

We use the Sigmoid function, which smoothly squishes any number between −∞ and +∞ into a range between 0 and 1:

In [15]:
sample_z = np.array([-5, 0, 8.85])
print("z values:", sample_z)
print("Probabilities:", np.round(sigmoid(sample_z), 4))

z values: [-5.    0.    8.85]
Probabilities: [0.0067 0.5    0.9999]


## 5. Putting it all together: Making Predictions

Now, let's use all your functions together. We will set up some dummy weights (since we haven't learned Gradient Descent yet), calculate the probabilities, and use probability_to_class function to make the final decision!

In [16]:
# We now have 7 features (age, tenure, charges, calls, and 3 contract types)
w_dummy = np.array([0.1, -0.4, 0.2, 0.5, 0.3, -0.2, -0.5])
b_dummy = 0.0

# 1. Calculate probabilities for the first 5 training customers
probs = predict_probability(X_train_scaled[:5], w_dummy, b_dummy)

# 2. Convert probabilities to final class predictions (threshold = 0.5)
predictions = probability_to_class(probs, threshold=0.5)

print("Customer | Probability | Prediction | Actual")
print("-" * 46)
for i in range(5):
    p = probs[i]
    pred = predictions[i]
    actual = y_train_num[i]
    print(f"   {i}     |    {p:.4f}   |      {pred}     |   {actual}")

Customer | Probability | Prediction | Actual
----------------------------------------------
   0     |    0.4789   |      0     |   0
   1     |    0.1927   |      0     |   0
   2     |    0.0980   |      0     |   0
   3     |    0.1547   |      0     |   0
   4     |    0.1655   |      0     |   0


## 1. What Does Training Mean?

In Day 2, we used manually selected dummy weights to make predictions.

Those weights were not learned from the dataset.

Training means finding weights and a bias that make the model's predictions increasingly better.

The basic training cycle is:

Initial weights
↓
Prediction
↓
Calculate loss
↓
Calculate gradients
↓
Update weights and bias
↓
Repeat

The Logistic Regression prediction is:

z = wᵀx + b

p = σ(z)

Day 3 will now explain how the model measures its prediction error and uses that error to learn better parameters.

## 2. Binary Cross-Entropy Loss

After the model makes a probability prediction, we need to measure how wrong that prediction is.

For binary classification, we use Binary Cross-Entropy (BCE), also called Log Loss.

For one example:

L = -[y log(p) + (1-y) log(1-p)]

Where:

- y = actual class (0 or 1)
- p = predicted probability
- L = loss

If y = 1:

L = -log(p)

If y = 0:

L = -log(1-p)

A correct and confident prediction gives a small loss.

A wrong and confident prediction gives a large loss.

The goal of training is to find weights and bias that minimize the loss.

## 3. Understanding Loss with Numbers

Let's calculate Binary Cross-Entropy loss for different predicted probabilities.

For this experiment, the actual class is:

y = 1

Therefore:

L = -log(p)

A prediction close to 1 should produce a small loss.

A prediction close to 0 should produce a large loss.

In [17]:
import numpy as np

y = 1

probabilities = np.array([0.99, 0.9, 0.7, 0.5, 0.1, 0.01])

losses = -(y * np.log(probabilities) + (1 - y) * np.log(1 - probabilities))

for p, loss in zip(probabilities, losses):
    print(f"Probability = {p:.2f} → Loss = {loss:.4f}")

Probability = 0.99 → Loss = 0.0101
Probability = 0.90 → Loss = 0.1054
Probability = 0.70 → Loss = 0.3567
Probability = 0.50 → Loss = 0.6931
Probability = 0.10 → Loss = 2.3026
Probability = 0.01 → Loss = 4.6052


### Actual Class = 0

Now suppose the customer did NOT churn:

y = 0

The BCE formula becomes:

L = -log(1-p)

Therefore, a prediction close to 0 should have a small loss, while a prediction close to 1 should have a large loss.

In [18]:
y = 0

probabilities = np.array([0.01, 0.1, 0.3, 0.5, 0.9, 0.99])

losses = -(y * np.log(probabilities) + (1 - y) * np.log(1 - probabilities))

for p, loss in zip(probabilities, losses):
    print(f"Probability = {p:.2f} → Loss = {loss:.4f}")

Probability = 0.01 → Loss = 0.0101
Probability = 0.10 → Loss = 0.1054
Probability = 0.30 → Loss = 0.3567
Probability = 0.50 → Loss = 0.6931
Probability = 0.90 → Loss = 2.3026
Probability = 0.99 → Loss = 4.6052


## 5. Derivative of the Sigmoid Function

The sigmoid function is:

σ(z) = 1 / (1 + e⁻ᶻ)

We need its derivative because our Logistic Regression model has the following chain:

w → z → p → L

where:

z = wᵀx + b

p = σ(z)

L = Binary Cross-Entropy Loss

Therefore, we need to know how p changes when z changes:

dp/dz

### Step 1 — Rewrite the sigmoid

σ(z) = (1 + e⁻ᶻ)⁻¹

Using the chain rule:

dσ/dz
= -(1 + e⁻ᶻ)⁻² × (-e⁻ᶻ)

Therefore:

dσ/dz = e⁻ᶻ / (1 + e⁻ᶻ)²

### Step 2 — Rewrite the derivative using the sigmoid itself

We have:

dσ/dz = e⁻ᶻ / (1 + e⁻ᶻ)²

Split the fraction:

dσ/dz
= [1 / (1 + e⁻ᶻ)] [e⁻ᶻ / (1 + e⁻ᶻ)]

The first term is the sigmoid:

1 / (1 + e⁻ᶻ) = σ(z)

For the second term:

e⁻ᶻ / (1 + e⁻ᶻ)

= [(1 + e⁻ᶻ) - 1] / (1 + e⁻ᶻ)

= 1 - 1/(1 + e⁻ᶻ)

= 1 - σ(z)

Therefore:

dσ/dz = σ(z)(1 - σ(z))

### Final Result

Therefore, the derivative of the sigmoid is:

dσ/dz = σ(z)(1 - σ(z))

If:

p = σ(z)

then we can write:

dp/dz = p(1-p)

### Why Does the Derivative Have This Form?

The derivative does NOT equal the sigmoid simply because we chose it that way.

We first differentiate:

σ(z) = 1/(1 + e⁻ᶻ)

The derivative is initially:

dσ/dz = e⁻ᶻ/(1 + e⁻ᶻ)²

Then algebraically rewrite it using the original sigmoid function.

This produces:

dσ/dz = σ(z)(1 - σ(z))

This is a special and useful property of the logistic sigmoid.

### Why Do We Need This in Logistic Regression?

Our model is:

z = wᵀx + b

Then:

p = σ(z)

Then:

L = -[y log(p) + (1-y)log(1-p)]

The dependency chain is:

w → z → p → L

Using the chain rule:

∂L/∂w
=
(∂L/∂p)
(∂p/∂z)
(∂z/∂w)

We now know:

∂p/∂z = p(1-p)

This will allow us to derive the complete Logistic Regression gradient.

### Important Mathematical Insight

The sigmoid derivative is largest around z = 0.

At z = 0:

σ(0) = 0.5

Therefore:

σ'(0) = 0.5(1 - 0.5)
      = 0.25

For very large positive or negative z:

σ'(z) → 0

Therefore, the sigmoid changes most rapidly around its center and becomes flatter toward its two extremes.

## 6. Gradient Descent

Gradient Descent is an iterative optimization algorithm used to find model parameters that minimize the loss.

Our Logistic Regression model calculates:

z = wᵀx + b

p = sigmoid(z)

Then we calculate the Binary Cross-Entropy loss.

The gradients are:

∂L/∂w = (p-y)x

∂L/∂b = p-y

Gradient Descent updates the parameters in the opposite direction of the gradient.

Weight update:

w := w - α(∂L/∂w)

Bias update:

b := b - α(∂L/∂b)

where α is the learning rate.

The learning rate controls the size of each parameter update.

A large learning rate makes large updates.
A small learning rate makes small updates.

The training process is repeated for multiple epochs.

One epoch means one complete pass through the training dataset.

Complete learning loop:

X
↓
z = wᵀx + b
↓
p = sigmoid(z)
↓
loss
↓
gradient
↓
update w and b
↓
repeat

The goal is to repeatedly update the parameters so that the loss decreases and eventually stabilizes.

## 7. Batch Gradient and Averaging

For one training example, we derived:

∂L/∂w = (p-y)x

∂L/∂b = p-y

However, our model is trained using multiple examples.

For m training examples, each example produces its own gradient.

The gradient of the total cost is the average of the individual gradients:

∂J/∂w = (1/m) Σᵢ (pᵢ-yᵢ)xᵢ

∂J/∂b = (1/m) Σᵢ (pᵢ-yᵢ)

where m is the number of training examples.

For ChurnLab:

m = 16

Therefore, during batch gradient descent:

16 training examples
↓
calculate predictions
↓
calculate individual errors
↓
calculate individual gradients
↓
average the gradients
↓
update weights and bias

This is called Batch Gradient Descent because the complete training batch is used to calculate the gradient before updating the parameters.

The important idea is:

One customer → one gradient

All customers → average gradient

Average gradient → one parameter update

## 8. Complete Logistic Regression Training Loop

We now combine all the mathematical concepts learned in Day 3.

### Step 1 — Initialize

Start with initial weights and bias.

For our 7 input features:

w = [w₁, w₂, ..., w₇]

b = 0

The model has not learned these parameters yet.

### Step 2 — Linear Prediction

Calculate the linear score:

z = Xw + b

### Step 3 — Probability

Apply the sigmoid function:

p = σ(z)

where:

σ(z) = 1 / (1 + e⁻ᶻ)

The result is a probability between 0 and 1.

### Step 4 — Calculate Binary Cross-Entropy Loss

J = -(1/m) Σ [yᵢ log(pᵢ) + (1-yᵢ)log(1-pᵢ)]

The loss measures how different our predictions are from the actual labels.

### Step 5 — Calculate Gradients

Weight gradient:

∇wJ = (1/m) Xᵀ(p-y)

Bias gradient:

∂J/∂b = (1/m) Σ(p-y)

The gradients tell us how the parameters should change to reduce the loss.

### Step 6 — Update Parameters

Using the learning rate α:

w := w - α∇wJ

b := b - α(∂J/∂b)

### Step 7 — Repeat

Repeat the complete process for multiple epochs.

Complete training loop:

Initialize w and b
↓
Calculate z
↓
Calculate probability p
↓
Calculate loss J
↓
Calculate gradients
↓
Update w and b
↓
Repeat for the next epoch

One complete pass through the training data is called an epoch.

The goal of training is to find values of w and b that produce a low loss.

### Complete Mathematical Pipeline

X
↓
z = Xw + b
↓
p = sigmoid(z)
↓
Binary Cross-Entropy Loss
↓
Gradients
↓
Gradient Descent
↓
Updated w and b
↓
Repeat

In [19]:
from src.logistic_regression import LogisticRegression
model = LogisticRegression()

y_true = np.array([1, 0, 1, 1])
probabilities = np.array([0.9, 0.2, 0.8, 0.7])

loss = model.binary_cross_entropy(y_true, probabilities)

print("Binary Cross-Entropy Loss:", loss)

Binary Cross-Entropy Loss: 0.22708064055624455


In [20]:
import importlib
import src.logistic_regression as lr

importlib.reload(lr)

from src.logistic_regression import LogisticRegression

In [21]:
print(hasattr(LogisticRegression, "calculate_gradients"))

True


In [22]:
X_sample = np.array([
    [1.0, 2.0],
    [2.0, 1.0],
    [3.0, 4.0]
])

y_sample = np.array([1, 0, 1])

probabilities_sample = np.array([0.8, 0.3, 0.9])

model = LogisticRegression()

weight_gradient, bias_gradient = model.calculate_gradients(
    X_sample,
    y_sample,
    probabilities_sample
)

print("Weight gradient:", weight_gradient)
print("Bias gradient:", bias_gradient)

Weight gradient: [ 0.03333333 -0.16666667]
Bias gradient: 1.850371707708594e-17


In [23]:
import numpy as np

from src.logistic_regression import LogisticRegression

In [24]:
import numpy as np
from src.logistic_regression import LogisticRegression

In [25]:
from pathlib import Path

path = Path(r"D:\churnlab\src\logistic_regression.py")

print(path.read_text())

import numpy as np


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


def predict_probability(X, w, b):
    z = np.dot(X, w) + b
    return sigmoid(z)


def probability_to_class(probabilities, threshold=0.5):
    # If probability is greater than or equal to the threshold,
    # predict class 1. Otherwise, predict class 0.
    return (probabilities >= threshold).astype(int)


class LogisticRegression:

    def __init__(self, learning_rate=0.01, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0
        self.loss_history = []

    def binary_cross_entropy(self, y_true, probabilities):
        epsilon = 1e-15

        probabilities = np.clip(probabilities, epsilon, 1 - epsilon) ##Keep every probability between epsilon and 1-epsilon

        loss = -np.mean(
            y_true * np.log(probabilities) + (1 - y_true) * np.log(1 - probabilities)
        )

        return loss

    def calculate_gradients(sel

In [26]:
model = lr.LogisticRegression(learning_rate=0.1)

print(hasattr(model, "update_parameters"))

True


In [27]:
model.weights = np.array([0.5, -0.5])
model.bias = 0.2

weight_gradient = np.array([0.1, -0.2])
bias_gradient = 0.05

print("Before update:")
print("Weights:", model.weights)
print("Bias:", model.bias)

model.update_parameters(weight_gradient, bias_gradient)

print("\nAfter update:")
print("Weights:", model.weights)
print("Bias:", model.bias)

Before update:
Weights: [ 0.5 -0.5]
Bias: 0.2

After update:
Weights: [ 0.49 -0.48]
Bias: 0.195


In [28]:
import importlib
import src.logistic_regression as lr

importlib.reload(lr)

print(lr.LogisticRegression.__dict__.keys())

dict_keys(['__module__', '__firstlineno__', '__init__', 'binary_cross_entropy', 'calculate_gradients', 'update_parameters', 'fit', '__static_attributes__', '__dict__', '__weakref__', '__doc__'])


In [29]:
model = lr.LogisticRegression(
    learning_rate=0.1,
    epochs=10
)

print(hasattr(model, "fit"))

True


In [30]:
X_sample = np.array([
    [0, 0],
    [0, 1],
    [1, 0],
    [1, 1]
], dtype=float)

y_sample = np.array([0, 0, 0, 1])

model.fit(X_sample, y_sample)

Epoch 1: Loss = 0.6931471805599453
Epoch 2: Loss = 0.6869753035255248
Epoch 3: Loss = 0.6811042900600279
Epoch 4: Loss = 0.6755119701953223
Epoch 5: Loss = 0.6701778403683764
Epoch 6: Loss = 0.665082952905746
Epoch 7: Loss = 0.6602098093485604
Epoch 8: Loss = 0.655542258287355
Epoch 9: Loss = 0.6510653981535127
Epoch 10: Loss = 0.6467654852342619


In [31]:
print("Learned weights:", model.weights)
print("Learned bias:", model.bias)

Learned weights: [0.01251182 0.01251182]
Learned bias: -0.22454510800407246


In [32]:
probabilities = lr.predict_probability(
    X_sample,
    model.weights,
    model.bias
)

predictions = lr.probability_to_class(probabilities)

print("Probabilities:", np.round(probabilities, 4))
print("Predictions:", predictions)
print("Actual:", y_sample)

Probabilities: [0.4441 0.4472 0.4472 0.4503]
Predictions: [0 0 0 0]
Actual: [0 0 0 1]


In [33]:
model = lr.LogisticRegression(
    learning_rate=0.1,
    epochs=1000
)

model.fit(X_sample, y_sample)

Epoch 1: Loss = 0.6931471805599453
Epoch 2: Loss = 0.6869753035255248
Epoch 3: Loss = 0.6811042900600279
Epoch 4: Loss = 0.6755119701953223
Epoch 5: Loss = 0.6701778403683764
Epoch 6: Loss = 0.665082952905746
Epoch 7: Loss = 0.6602098093485604
Epoch 8: Loss = 0.655542258287355
Epoch 9: Loss = 0.6510653981535127
Epoch 10: Loss = 0.6467654852342619
Epoch 11: Loss = 0.6426298470350751
Epoch 12: Loss = 0.6386468010013477
Epoch 13: Loss = 0.6348055785252913
Epoch 14: Loss = 0.6310962540995979
Epoch 15: Loss = 0.6275096794326909
Epoch 16: Loss = 0.6240374223079123
Epoch 17: Loss = 0.6206717099479642
Epoch 18: Loss = 0.6174053766339185
Epoch 19: Loss = 0.6142318153231485
Epoch 20: Loss = 0.6111449330109706
Epoch 21: Loss = 0.6081391095852892
Epoch 22: Loss = 0.6052091599310124
Epoch 23: Loss = 0.6023502990505911
Epoch 24: Loss = 0.5995581099780328
Epoch 25: Loss = 0.5968285142756213
Epoch 26: Loss = 0.5941577449148898
Epoch 27: Loss = 0.5915423213558539
Epoch 28: Loss = 0.5889790266508452
Epo

In [34]:
print("Learned weights:", model.weights)
print("Learned bias:", model.bias)

probabilities = lr.predict_probability(
    X_sample,
    model.weights,
    model.bias
)

predictions = lr.probability_to_class(probabilities)

print("Probabilities:", np.round(probabilities, 4))
print("Predictions:", predictions)
print("Actual:", y_sample)

Learned weights: [3.03763559 3.03763559]
Learned bias: -4.773702236905248
Probabilities: [0.0084 0.1498 0.1498 0.7861]
Predictions: [0 0 0 1]
Actual: [0 0 0 1]


In [35]:
model = lr.LogisticRegression(
    learning_rate=0.1,
    epochs=1000
)

model.fit(X_sample, y_sample)


Epoch 1: Loss = 0.6931471805599453
Epoch 2: Loss = 0.6869753035255248
Epoch 3: Loss = 0.6811042900600279
Epoch 4: Loss = 0.6755119701953223
Epoch 5: Loss = 0.6701778403683764
Epoch 6: Loss = 0.665082952905746
Epoch 7: Loss = 0.6602098093485604
Epoch 8: Loss = 0.655542258287355
Epoch 9: Loss = 0.6510653981535127
Epoch 10: Loss = 0.6467654852342619
Epoch 11: Loss = 0.6426298470350751
Epoch 12: Loss = 0.6386468010013477
Epoch 13: Loss = 0.6348055785252913
Epoch 14: Loss = 0.6310962540995979
Epoch 15: Loss = 0.6275096794326909
Epoch 16: Loss = 0.6240374223079123
Epoch 17: Loss = 0.6206717099479642
Epoch 18: Loss = 0.6174053766339185
Epoch 19: Loss = 0.6142318153231485
Epoch 20: Loss = 0.6111449330109706
Epoch 21: Loss = 0.6081391095852892
Epoch 22: Loss = 0.6052091599310124
Epoch 23: Loss = 0.6023502990505911
Epoch 24: Loss = 0.5995581099780328
Epoch 25: Loss = 0.5968285142756213
Epoch 26: Loss = 0.5941577449148898
Epoch 27: Loss = 0.5915423213558539
Epoch 28: Loss = 0.5889790266508452
Epo

In [36]:
print("Number of recorded losses:", len(model.loss_history))

print("First 5 losses:")
print(np.round(model.loss_history[:5], 4))

print("\nLast 5 losses:")
print(np.round(model.loss_history[-5:], 4))

Number of recorded losses: 1000
First 5 losses:
[0.6931 0.687  0.6811 0.6755 0.6702]

Last 5 losses:
[0.144  0.1438 0.1437 0.1436 0.1435]


In [37]:
print("X_train_scaled shape:", X_train_scaled.shape)
print("y_train_num shape:", y_train_num.shape)

print("\nFirst training sample:")
print(X_train_scaled[0])

print("\nFirst training label:")
print(y_train_num[0])

X_train_scaled shape: (16, 7)
y_train_num shape: (16,)

First training sample:
[-0.09819034 -0.46785269  0.08441277 -0.17428904 -0.77459667  1.73205081
 -0.77459667]

First training label:
0


# Day 5 — Final Evaluation, Visualization, and Application

Today we connect the trained Logistic Regression model to the final evaluation pipeline and the Streamlit application.

The goal is not to introduce a new machine-learning algorithm. The goal is to understand how a trained model becomes a complete usable project.


## Day 5 Learning Goals

By the end of Day 5, we should understand:

1. How probabilities become class predictions.
2. Why accuracy, precision, recall, and F1 score are different.
3. How the confusion matrix summarizes classification results.
4. How the loss history shows the training process.
5. How learned weights can be interpreted as feature effects in the model.
6. How the complete pipeline moves from raw customer data to a final prediction.
7. How the same trained pipeline is connected to the Streamlit application.


## 1. Probability → Class Prediction

Logistic Regression first produces a probability between 0 and 1.

For example, a probability of 0.82 means the model estimates an 82% probability for class 1.

The threshold converts the probability into a class:

- probability ≥ 0.5 → class 1
- probability < 0.5 → class 0

The threshold is a decision rule; it is not part of the sigmoid calculation itself.


In [38]:
from src.logistic_regression import probability_to_class

example_probabilities = np.array([0.12, 0.48, 0.50, 0.91])

example_predictions = probability_to_class(example_probabilities)

print("Probabilities:", example_probabilities)
print("Predictions:", example_predictions)


Probabilities: [0.12 0.48 0.5  0.91]
Predictions: [0 0 1 1]


## 2. Confusion Matrix

For binary classification, compare the actual label with the predicted label.

- True Positive (TP): actual 1, predicted 1
- True Negative (TN): actual 0, predicted 0
- False Positive (FP): actual 0, predicted 1
- False Negative (FN): actual 1, predicted 0

These four counts are the foundation for the main classification metrics.


In [39]:
actual = np.array([0, 0, 1, 1, 1, 0])
predicted = np.array([0, 1, 1, 0, 1, 0])

tp = np.sum((actual == 1) & (predicted == 1))
tn = np.sum((actual == 0) & (predicted == 0))
fp = np.sum((actual == 0) & (predicted == 1))
fn = np.sum((actual == 1) & (predicted == 0))

print("TP:", tp)
print("TN:", tn)
print("FP:", fp)
print("FN:", fn)


TP: 2
TN: 2
FP: 1
FN: 1


## 3. Evaluation Metrics

Accuracy asks: how many predictions were correct overall?

Precision asks: among customers predicted as churners, how many actually churned?

Recall asks: among customers who actually churned, how many did the model identify?

F1 score combines precision and recall using their harmonic mean.

These metrics answer different questions, so one metric should not automatically be treated as the complete evaluation of a classifier.


In [40]:
from src.metrics import accuracy, precision, recall, f1_score

print("Accuracy:", accuracy(actual, predicted))
print("Precision:", precision(actual, predicted))
print("Recall:", recall(actual, predicted))
print("F1 Score:", f1_score(actual, predicted))


Accuracy: 0.6666666666666666
Precision: 0.6666666666666666
Recall: 0.6666666666666666
F1 Score: 0.6666666666666666


## 4. Training Loss

During training, Logistic Regression records the loss after each epoch.

A decreasing loss generally means the optimization process is finding parameters that fit the training data better.

The loss history is useful because it lets us inspect the learning process rather than looking only at the final weights.


In [41]:
if hasattr(model, "loss_history"):
    print("Number of recorded losses:", len(model.loss_history))
    print("First 5 losses:", np.round(model.loss_history[:5], 4))
    print("Last 5 losses:", np.round(model.loss_history[-5:], 4))


Number of recorded losses: 1000
First 5 losses: [0.6931 0.687  0.6811 0.6755 0.6702]
Last 5 losses: [0.144  0.1438 0.1437 0.1436 0.1435]


## 5. Learned Weights

After training, the model contains one learned weight for every input feature.

The weight is part of the linear score z = Xw + b. Its sign and magnitude influence the score, but interpretation should be made with awareness that the numerical features were standardized and the categorical feature was encoded.

A large positive weight pushes the linear score upward when that feature increases by one standardized unit. A negative weight pushes it downward.


In [42]:
feature_names = list(X_train_encoded.columns)

for feature, weight in zip(feature_names, model.weights):
    print(f"{feature}: {weight:.4f}")

print("Bias:", model.bias)


age: 3.0376
tenure: 3.0376
Bias: -4.773702236905248


## 6. Complete Day 5 Pipeline

The complete project now follows this flow:

Raw CSV
↓
Load data
↓
Separate X and y
↓
Train/test split
↓
Categorical encoding
↓
Convert to NumPy
↓
Calculate training mean and standard deviation
↓
Standardize training and test data
↓
Logistic Regression training
↓
Learn weights and bias
↓
Predict test probabilities
↓
Convert probabilities to classes
↓
Calculate evaluation metrics
↓
Visualize loss and learned weights
↓
Use the same pipeline inside the Streamlit application


## 7. Why the Streamlit App Reuses the Same Pipeline

The application should not create a different machine-learning process.

A new customer's data must go through the same feature structure and scaling logic used during training. In particular, the training mean and training standard deviation are reused when standardizing new input.

The application then uses the learned weights and bias to calculate a probability and applies the classification threshold to produce the final prediction.


## Final Day 5 Takeaway

The important result of this project is not only that the model predicts churn.

The project demonstrates the complete reasoning pipeline of a binary classification system: preprocessing → mathematical model → optimization → prediction → evaluation → visualization → application.
